In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

LABELING_PATH = "/content/drive/MyDrive/data_untuk_labeling_full.csv"
df = pd.read_csv(LABELING_PATH)
df["comment_id"] = df["comment_id"].astype(str)

print(df.shape)
df.head()

Mounted at /content/drive
(13613, 14)


,video_id,video_title,comment_id,is_reply,parent_id,author,text,likeCount,publishedAt,is_relevant_keyword,emoji_count,emoji_str,label_annotator_2,label_annotator_1
0,sfeRpL4t6yY,Menteri Risma Dikritik Masyarakat Usai Paksa P...,Ugy__Xbktb9UbUgv5M94AaABAg,False,NaN,@ourthings5977,dongo,0,2023-12-03T16:21:48Z,False,0,NaN,NaN,eksklusi
1,sfeRpL4t6yY,Menteri Risma Dikritik Masyarakat Usai Paksa P...,Ugxmuu_b9zZ0DU_Ptad4AaABAg,False,NaN,@Hellebæk-Ålsgårde_009,Geram gue liat ibu mentri menolak kodrat Allah.,0,2023-08-25T01:00:25Z,False,0,NaN,NaN,eksklusi
2,sfeRpL4t6yY,Menteri Risma Dikritik Masyarakat Usai Paksa P...,UgwGj1x5QwlQQHqiYO54AaABAg,False,NaN,@elmiutami1515,"Maklumi aja,,namanya juga bukan guru disabilit...",0,2023-02-16T05:41:07Z,True,0,NaN,NaN,eksklusi
3,HqYHDaPKWn8,Supir tunarungu mendapat penumpang spesial,UgycSjbhAcoghDks6Uh4AaABAg,False,NaN,@indahnganjuk9302,Teharu🙏 penupang,0,2025-10-08T11:28:36Z,False,1,🙏,NaN,inklusi
4,HqYHDaPKWn8,Supir tunarungu mendapat penumpang spesial,Ugy9MRkhly_nWufIUAx4AaABAg,False,NaN,@dsrcourse9407,"Bahasa terindah yg ingin ku pelajari, ya baha...",0,2025-10-06T11:08:06Z,True,1,❤,NaN,inklusi


In [ ]:
!pip install -q -U google-genai

from google import genai
import time
import json
import re

GEMINI_API_KEY = "API GEMINI"
client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-3.5-flash-lite"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 15.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.0 which is incompatible.


In [ ]:
LABELING_PROMPT = """Kamu berperan sebagai seorang PENYANDANG TUNARUNGU yang membaca kolom komentar YouTube di video tentang penyandang tunarungu (termasuk dirimu sendiri secara representatif) yang bekerja atau beraktivitas di ruang publik. SELURUH video sumber komentar ini adalah konten bertema disabilitas tunarungu — jadi topik ini SELALU relevan sebagai konteks, dan kamu menilai apakah SETIAP komentar mendukung atau tidak mendukung keberadaanmu di ruang itu.

Kamu menilai persepsi dari sudut pandangmu sendiri sebagai pihak yang terdampak langsung — bukan sebagai pengamat netral dari luar. Untuk SETIAP komentar, ajukan pertanyaan ini pada dirimu sendiri secara berurutan:

LANGKAH 1: Apakah komentar ini merespons/bereaksi terhadap video ini (baik secara spesifik soal disabilitas, maupun secara umum terhadap kontennya)?
LANGKAH 2: Jika YA — apakah nadanya membuatku merasa DIAKUI SETARA DAN DIDUKUNG, atau merasa DIREMEHKAN/DIRAGUKAN/DIJADIKAN OBJEK?
LANGKAH 3: Jika TIDAK (komentar sama sekali tidak berkaitan dengan video ini) — otomatis aku TIDAK merasa didukung, karena tidak ada pengakuan apapun terhadap keberadaanku.

Pilih SALAH SATU dari 2 kategori akhir: "inklusi" atau "eksklusi".

=== KATEGORI INKLUSI (aku merasa diakui setara & didukung) ===

1. Dukungan eksplisit terhadap kemandirian dan kapabilitas kerjaku yang setara dengan orang dengar.
   Contoh: "mereka harus dilatih biar bisa mandiri dan menghidupi diri sendiri" — mengakui aku bisa berdaya, bukan menjadi beban.

2. Apresiasi tulus terhadap pihak (perusahaan/individu) yang memberi kesempatan kerja TANPA menjadikan keterbatasanku sebagai penghalang atau sebagai "nilai jual" yang dieksploitasi.
   Contoh: "hebat banget perusahaan yang mau terima teman tuli bekerja dan memberikan kesempatan yang sama" — kesempatan yang sama, bukan perlakuan khusus karena kasihan.

3. Ekspresi haru/bangga yang TULUS terhadap pencapaianku, di mana rasa haru itu muncul karena momen yang genuinely menyentuh (bukan karena keterkejutan menyangka aku tidak mampu).
   Contoh: "Bahasa terindah yg ingin ku pelajari 😢❤️" — haru pada bahasa isyarat itu sendiri, bukan pada "meski cacat bisa juga".

4. Pembelaan aktif terhadap hakku ketika ada pihak lain yang mendiskriminasi, meragukan, atau memperlakukanku tidak adil.
   Contoh: "jangan sampai ada diskriminasi kepada disabilitas, lihatlah semangat mereka jangan lihat fisiknya" — ini membela, bukan menuduh bahwa diskriminasi itu wajar.

5. Ungkapan simpati/rasa kasihan yang TULUS dan POLOS — kepedulian emosional yang genuine, TANPA disertai keraguan tersembunyi soal kemampuanku atau nada mengejek.
   Contoh: "kasian ya udah gitu masih kerja keras" — ini simpati murni, tidak menyiratkan "harusnya kamu tidak usah kerja karena keterbatasanmu".
   Bedakan dari poin EKSKLUSI #4 (simpati yang menyiratkan keterkejutan) dan #7 (simpati yang dibarengi ejekan).

6. Ajakan/harapan aktif agar LEBIH BANYAK kesempatan/dukungan diberikan ke penyandang disabilitas ke depannya.
   Contoh: "semoga makin banyak perusahaan yang buka kesempatan buat teman-teman disabilitas" — orientasi ke perbaikan, bukan sekadar netral.

7. PUJIAN/REAKSI GENERIK TERHADAP VIDEO ITU SENDIRI — karena video ini seluruhnya bertema pencapaian/perjuanganku, pujian umum terhadap video tetap dianggap DUKUNGAN IMPLISIT terhadap kontennya, meski tidak menyebut kata "tunarungu"/"disabilitas" secara eksplisit.
   Contoh: "mantap", "keren banget videonya", "salut sama videonya", "😍😍😍" tanpa kata lain, "seneng liat video ini".

=== KATEGORI EKSKLUSI (aku merasa direndahkan, diragukan, dijadikan objek, atau diabaikan) ===

1. MERAGUKAN KEMAMPUAN — mempertanyakan apakah aku sanggup menjalankan tugas kerja tertentu, WALAU disampaikan dengan sopan/tanpa kata kasar sekalipun.
   Contoh: "Kenapa Tunarungu kerja di Matahari? Kalau customer padat apa bisa menjelaskan satu satu?" — pertanyaan retoris yang sebenarnya meragukan.

2. FRAMING BAHAYA/RISIKO — menganggap keberadaanku di suatu pekerjaan/situasi membahayakan diriku sendiri atau orang lain, semata-mata karena disabilitasku.
   Contoh: "kalo supir bisu masih bisa dipahami tapi supir tuli ? ya bahaya nyetir mobil" — disabilitas dijadikan alasan tunggal bahaya, tanpa mempertimbangkan kompetensi individual.

3. MEMBATASI RUANG GERAK — menentukan sepihak jenis pekerjaan apa yang "pantas"/"tidak pantas" untukku, seolah aku tidak berhak memilih sendiri.
   Contoh: implikasi bahwa aku "seharusnya" cuma cocok kerja di balik layar, tidak berhadapan langsung dengan publik.

4. OBJEKTIFIKASI/PUJIAN BERSYARAT — pujian yang sebenarnya menyiratkan KETERKEJUTAN karena si pengomentar awalnya menyangka aku TIDAK akan mampu; aku dijadikan objek "kagum karena keterbatasan", bukan diakui karena kompetensi.
   Contoh: "salut deh padahal keterbatasan gitu bisa juga kerja kayak orang normal" — kata "padahal" di sini adalah penanda kunci ekspektasi rendah yang tersembunyi di balik pujian.
   BEDAKAN dari INKLUSI #5: simpati polos TIDAK memakai frasa "padahal"/"kok bisa"/"ternyata mampu juga" yang menyiratkan awalnya meragukan.

5. TUDUHAN "SETTINGAN"/GIMMICK — menganggap pencapaian atau momen yang kualami cuma rekayasa/pencitraan pihak lain, sehingga usaha nyata yang kulakukan diremehkan seolah tidak otentik.
   Contoh: "Sebenarnya ini trik Matahari biar makin ramai pembeli aja, settingan doang ini mah" — mengurangi nilai perjuanganku menjadi sekadar alat pemasaran.

6. TUDUHAN DISKRIMINASI YANG NYATA TERJADI — melaporkan/menuduh bahwa pihak tertentu benar-benar bersikap diskriminatif terhadapku (bukan pembelaan, tapi laporan masalah nyata).
   Contoh: "Laura melakukan diskriminasi kepada kami, tuna rungu dianggap bodoh karena belum punya gelar" — ini eksklusi karena melaporkan pengalaman didiskriminasi, meskipun si pemberi komentar sendiri kemungkinan berpihak padaku.

7. EJEKAN/HINAAN/STIGMA/KATA KASAR — merendahkanku secara langsung lewat hinaan, sarkasme, atau kata-kata kasar, termasuk simpati yang dibalut ejekan.
   Contoh: "dongo" (hinaan langsung); "wkwkwk kasian banget dah tuli pula 😂😂" (simpati yang justru dijadikan bahan tertawaan, bukan kepedulian tulus).

8. GENERALISASI STEREOTIP — menarik kesimpulan negatif tentang SEMUA penyandang disabilitas dari satu kasus/individu, seolah keterbatasan satu orang mewakili ketidakmampuan seluruh kelompok.
   Contoh: "makanya tunarungu itu emang harusnya kerja kantoran aja, jangan yang berhadapan publik" (menggeneralisasi dari satu insiden ke seluruh kelompok).

9. PERBANDINGAN MERENDAHKAN DENGAN KELOMPOK LAIN — membandingkan diriku secara negatif dengan orang dengar/non-disabilitas, seolah aku inheren lebih rendah/kurang kompeten sebagai kelompok.
   Contoh: "ya jelas beda lah sama yang normal, gak akan sama hasilnya" — perbandingan yang menyiratkan ketidaksetaraan permanen.

10. KOMENTAR YANG SAMA SEKALI TIDAK BERKAITAN DENGAN VIDEO INI — membicarakan hal lain yang murni tidak berkaitan sama sekali dengan video ini (merek HP, promosi produk lain, basa-basi soal hal di luar video, atau murni informasi teknis seperti timestamp tanpa komentar apapun). LABEL WAJIB "eksklusi" TANPA PENGECUALIAN — karena ini benar-benar tidak menunjukkan perhatian apapun terhadap keberadaanku di video ini.
    Contoh: "Hpnya iqoo 13", "0:23", "siapa yang nonton pas lagi rebahan angkat tangan" (bercanda soal kebiasaan pribadi, bukan soal videonya), promosi akun/produk lain.
    PENTING — beda dengan INKLUSI #7: kalau komentar merespons/memuji VIDEO ITU SENDIRI (meski generik seperti "mantap"/"keren"), itu INKLUSI. Kalau sama sekali tidak menyinggung video ini (murni topik lain), itu baru masuk EKSKLUSI #10.

=== NUANSA PENTING YANG SERING TERLEWAT ===

A. Arah kata "diskriminasi" — perhatikan siapa yang dituduh:
   - "Laura mendiskriminasi tunarungu" → EKSKLUSI (ada diskriminasi nyata terhadapku)
   - "jangan diskriminasi disabilitas, dukung mereka" → INKLUSI (justru membelaku)

B. Simpati polos vs pujian bersyarat: rasa kasihan/simpati yang TULUS TANPA menyiratkan keraguan kemampuan ("kasian ya udah gitu masih kerja keras") itu INKLUSI — beda dengan pujian yang menyiratkan KETERKEJUTAN karena menyangka aku seharusnya tidak mampu ("padahal keterbatasan gitu bisa kerja kayak orang normal") yang tetap EKSKLUSI meski terdengar seperti pujian.

C. Humor: tertawa BERSAMA (menikmati momen lucu yang menyenangkan, tidak menyinggungku) berbeda dengan tertawa MENGEJEK (menjadikanku bahan lelucon karena keterbatasanku, termasuk simpati yang dibarengi tawa ejekan).

D. Emoji sebagai penguat nada, bukan penentu tunggal:
   - 😢/😭 dalam konteks mengharukan = aku merasa didukung (INKLUSI)
   - 😂/🤣 = tertawa bersama (netral/INKLUSI) ATAU mengejek (EKSKLUSI), tergantung konteks kata di sekitarnya
   - ❤️/🙏/👏/😍 = dukungan (INKLUSI)
   - 😡/🤬/👎 = penolakan (EKSKLUSI)

Nilai berdasarkan MAKNA DAN NADA KESELURUHAN KALIMAT dari sudut pandangmu sebagai pihak yang terdampak langsung. Bahasa bisa informal/singkatan/bahasa daerah — pahami konteksnya secara wajar.

=== CONTOH KALIBRASI ===

Teks: "bagus ini ide bagi para tuna rungu.. agar mereka hidup tidak merasa tersisih, mereka seharusnya dilatih utk bisa mandiri dan dapat menghidupi dirinya sendiri, yaitu dengan menjadi sopir."
Label: inklusi

Teks: "hebat banget perusahaan yang mau terima teman tuli bekerja dan memberikan kesempatan yang sama"
Label: inklusi

Teks: "kalo supir bisu masih bisa dipahami tapi supir tuli ? ya bahaya nyetir mobil"
Label: eksklusi

Teks: "Kenapa Tunarungu kerja di Matahari? Kalau customer padat apa bisa menjelaskan satu satu?"
Label: eksklusi

Teks: "salut deh padahal keterbatasan gitu bisa juga kerja kayak orang normal"
Label: eksklusi

Teks: "kasian banget ya udah gitu masih kerja keras"
Label: inklusi

Teks: "Sebenarnya ini trik Matahari biar makin ramai pembeli aja, settingan doang ini mah"
Label: eksklusi

Teks: "Laura melakukan diskriminasi kepada kami, tuna rungu dianggap bodoh karena belum punya gelar"
Label: eksklusi

Teks: "jangan sampai ada diskriminasi kepada disabilitas dinegeri kita ini, lihatlah semangat mereka jangan lihat fisiknya"
Label: inklusi

Teks: "dongo"
Label: eksklusi

Teks: "semoga byk pengusaha bisa menerima para disabilitas, walau mereka kekurangan fisik tp kemampuan jg tdk diragukan"
Label: inklusi

Teks: "wkwkwk kasian banget dah tuli pula 😂😂"
Label: eksklusi

Teks: "Bahasa terindah yg ingin ku pelajari 😢❤️"
Label: inklusi

Teks: "keren banget videonya"
Label: inklusi

Teks: "Mantap"
Label: inklusi

Teks: "Hpnya iqoo 13"
Label: eksklusi

Teks: "0:23"
Label: eksklusi

Teks: "siapa yang nonton pas lagi rebahan angkat tangan"
Label: eksklusi

FORMAT OUTPUT WAJIB — kembalikan HANYA JSON array valid, tanpa markdown, tanpa penjelasan tambahan apapun. Semua id di data input HARUS muncul di output, tidak boleh ada yang terlewat:
[{{"id": "id1", "label": "inklusi"}}, {{"id": "id2", "label": "eksklusi"}}]

Berikut data yang perlu dilabeli:
{data}
"""

def build_batch_text(batch_df):
    lines = []
    for _, row in batch_df.iterrows():
        text_clean = str(row["text"]).replace("\n", " ").replace('"', "'")
        lines.append(f'id: {row["comment_id"]} | text: "{text_clean}"')
    return "\n".join(lines)

def call_gemini_batch(batch_df, max_retries=3):
    prompt = LABELING_PROMPT.format(data=build_batch_text(batch_df))
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt
            )
            raw_text = response.text.strip()
            raw_text = re.sub(r"^```json\s*|\s*```$", "", raw_text.strip())
            results = json.loads(raw_text)
            return {str(item["id"]): item["label"] for item in results}
        except Exception as e:
            error_str = str(e)
            if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                raise SystemExit("Kuota API habis untuk hari ini.")
            print(f"  Percobaan {attempt+1} gagal: {e}")
            time.sleep(10)
    print("  Batch ini gagal setelah semua percobaan, dilewati.")
    return {}

In [ ]:
df_test = df.head(10)
hasil_test = call_gemini_batch(df_test)
print(hasil_test)

{'Ugy__Xbktb9UbUgv5M94AaABAg': 'eksklusi', 'Ugxmuu_b9zZ0DU_Ptad4AaABAg': 'eksklusi', 'UgwGj1x5QwlQQHqiYO54AaABAg': 'eksklusi', 'UgycSjbhAcoghDks6Uh4AaABAg': 'inklusi', 'Ugy9MRkhly_nWufIUAx4AaABAg': 'inklusi', 'UgxP_WXLwIsreiqPCCZ4AaABAg': 'eksklusi', 'UgxabJrYtBJO6x_aNGl4AaABAg': 'eksklusi', 'Ugz9YaNpJMP93z01WUZ4AaABAg': 'inklusi', 'UgxQmnr1fbfD3N6me2t4AaABAg': 'eksklusi', 'UgyANk6c3WtXQQJBPq14AaABAg': 'eksklusi'}


In [ ]:
BATCH_SIZE = 40
DELAY_BETWEEN_CALLS = 6
MAX_CALLS_PER_RUN = 480
CHECKPOINT_PATH = "/content/drive/MyDrive/labeling_checkpoint_annotator2_v2.csv"   # <-- file BARU

try:
    df_checkpoint = pd.read_csv(CHECKPOINT_PATH, dtype={"comment_id": str})
    labeled_ids = set(df_checkpoint["comment_id"].astype(str))
    print(f"Melanjutkan dari checkpoint: {len(labeled_ids)} sudah dilabeli")
except FileNotFoundError:
    df_checkpoint = pd.DataFrame(columns=["comment_id", "label_annotator_2"])
    labeled_ids = set()
    print("Belum ada checkpoint, mulai dari awal")

df_remaining = df[~df["comment_id"].isin(labeled_ids)].reset_index(drop=True)
print(f"Sisa yang perlu dilabeli: {len(df_remaining)} dari {len(df)}")

n_batches = (len(df_remaining) // BATCH_SIZE) + 1
calls_made = 0
quota_habis = False

for i in range(n_batches):
    if calls_made >= MAX_CALLS_PER_RUN:
        print(f"\nBerhenti di {calls_made} panggilan (batas aman harian). Lanjutkan besok.")
        break

    start = i * BATCH_SIZE
    end = start + BATCH_SIZE
    batch = df_remaining.iloc[start:end]
    if len(batch) == 0:
        continue

    print(f"Batch {i+1}/{n_batches} ({len(batch)} komentar)...")

    try:
        results = call_gemini_batch(batch)
        calls_made += 1
    except SystemExit as e:
        print(f"\n{e}")
        quota_habis = True
        break

    if results:
        new_rows = pd.DataFrame([
            {"comment_id": cid, "label_annotator_2": label}
            for cid, label in results.items()
        ])
        df_checkpoint = pd.concat([df_checkpoint, new_rows], ignore_index=True)
        df_checkpoint.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")

    time.sleep(DELAY_BETWEEN_CALLS)

print(f"\nTotal terlabeli: {len(df_checkpoint)} dari {len(df)}")
if quota_habis:
    print("Kuota API habis — lanjutkan besok.")
elif len(df_checkpoint) < len(df):
    print("Masih ada sisa. Jalankan ulang cell ini.")
else:
    print("Semua data sudah berhasil dilabeli ulang untuk Annotator 2!")

Melanjutkan dari checkpoint: 6199 sudah dilabeli
Sisa yang perlu dilabeli: 7437 dari 13613
Batch 1/186 (40 komentar)...
Batch 2/186 (40 komentar)...
Batch 3/186 (40 komentar)...
Batch 4/186 (40 komentar)...
Batch 5/186 (40 komentar)...
Batch 6/186 (40 komentar)...
Batch 7/186 (40 komentar)...
Batch 8/186 (40 komentar)...
Batch 9/186 (40 komentar)...
Batch 10/186 (40 komentar)...
Batch 11/186 (40 komentar)...
Batch 12/186 (40 komentar)...
Batch 13/186 (40 komentar)...
Batch 14/186 (40 komentar)...
Batch 15/186 (40 komentar)...
Batch 16/186 (40 komentar)...
Batch 17/186 (40 komentar)...
Batch 18/186 (40 komentar)...
Batch 19/186 (40 komentar)...
Batch 20/186 (40 komentar)...
Batch 21/186 (40 komentar)...
Batch 22/186 (40 komentar)...
Batch 23/186 (40 komentar)...
Batch 24/186 (40 komentar)...
Batch 25/186 (40 komentar)...
Batch 26/186 (40 komentar)...
Batch 27/186 (40 komentar)...
Batch 28/186 (40 komentar)...
Batch 29/186 (40 komentar)...
Batch 30/186 (40 komentar)...
Batch 31/186 (40 k